# Inventory Management — Multivariate Time-Series Forecasting

## Objective

This stage converts demand forecasts into inventory management decisions.

The objective is to estimate:

- Forecasted Demand
- Demand Variability
- Safety Stock
- Reorder Point (ROP)
- Inventory Status

The forecasting results from the previous model comparison stage will be used as the demand forecasting input.

## Inventory Management Flow

```text
Demand Forecast
       ↓
Demand During Lead Time
       ↓
Safety Stock
       ↓
Reorder Point
       ↓
Inventory Decision

In [1]:
import pandas as pd
import numpy as np

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [2]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os

for root, dirs, files in os.walk("/content/drive/MyDrive"):
    for file in files:
        if file == "transformer_validation_forecasts.parquet":
            print("File found:")
            print(os.path.join(root, file))

File found:
/content/drive/MyDrive/transformer_validation_forecasts.parquet


In [4]:
FORECAST_PATH = "/content/drive/MyDrive/transformer_validation_forecasts.parquet"

forecast_df = pd.read_parquet(FORECAST_PATH)

print("Forecast data loaded successfully.")
print("Shape:", forecast_df.shape)

print("\nColumns:")
print(forecast_df.columns.tolist())

print("\nDate range:")
print(forecast_df["date"].min(), "to", forecast_df["date"].max())

print("\nUnique item-store series:",
      forecast_df[["item_id", "store_id"]].drop_duplicates().shape[0])

print("\nMissing values:")
print(forecast_df.isna().sum())

print("\nFirst 5 rows:")
print(forecast_df.head())

Forecast data loaded successfully.
Shape: (853720, 5)

Columns:
['item_id', 'store_id', 'date', 'actual_demand', 'forecast_demand']

Date range:
2016-03-28 00:00:00 to 2016-04-24 00:00:00

Unique item-store series: 30490

Missing values:
item_id            0
store_id           0
date               0
actual_demand      0
forecast_demand    0
dtype: int64

First 5 rows:
         item_id store_id       date  actual_demand  forecast_demand
0  HOBBIES_1_001     CA_1 2016-03-28            1.0              0.0
1  HOBBIES_1_001     CA_1 2016-03-29            0.0              0.0
2  HOBBIES_1_001     CA_1 2016-03-30            0.0              0.0
3  HOBBIES_1_001     CA_1 2016-03-31            0.0              0.0
4  HOBBIES_1_001     CA_1 2016-04-01            0.0              0.0


In [5]:
# Prepare data for inventory analysis

forecast_df["date"] = pd.to_datetime(forecast_df["date"])

# Ensure correct time-series ordering
forecast_df = forecast_df.sort_values(
    ["item_id", "store_id", "date"]
).reset_index(drop=True)

# Forecast error
forecast_df["forecast_error"] = (
    forecast_df["actual_demand"] - forecast_df["forecast_demand"]
)

# Absolute forecast error
forecast_df["absolute_error"] = (
    forecast_df["forecast_error"].abs()
)

print("Data prepared successfully.")

print("\nShape:", forecast_df.shape)

print("\nDate range:")
print(forecast_df["date"].min(), "to", forecast_df["date"].max())

print("\nForecast demand summary:")
print(forecast_df["forecast_demand"].describe())

print("\nActual demand summary:")
print(forecast_df["actual_demand"].describe())

print("\nForecast error summary:")
print(forecast_df["forecast_error"].describe())

print("\nZero forecast percentage:",
      (forecast_df["forecast_demand"] == 0).mean() * 100)

print("\nZero actual percentage:",
      (forecast_df["actual_demand"] == 0).mean() * 100)

Data prepared successfully.

Shape: (853720, 7)

Date range:
2016-03-28 00:00:00 to 2016-04-24 00:00:00

Forecast demand summary:
count    8.537200e+05
mean     1.796730e-07
std      9.715532e-05
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      6.335764e-02
Name: forecast_demand, dtype: float64

Actual demand summary:
count    853720.000000
mean          1.386433
std           3.594687
min           0.000000
25%           0.000000
50%           0.000000
75%           1.000000
max         204.000000
Name: actual_demand, dtype: float64

Forecast error summary:
count    853720.000000
mean          1.386433
std           3.594687
min          -0.063358
25%           0.000000
50%           0.000000
75%           1.000000
max         204.000000
Name: forecast_error, dtype: float64

Zero forecast percentage: 99.99953146230615

Zero actual percentage: 56.27840509769011


In [6]:
print("Forecast demand statistics:")
print(forecast_df["forecast_demand"].describe())

print("\nActual demand statistics:")
print(forecast_df["actual_demand"].describe())

print("\nSample non-zero forecasts:")
print(
    forecast_df[forecast_df["forecast_demand"] > 0]
    [["item_id", "store_id", "date", "actual_demand", "forecast_demand"]]
    .head(20)
)

print("\nNumber of non-zero forecasts:",
      (forecast_df["forecast_demand"] > 0).sum())

print("\nMaximum forecast:",
      forecast_df["forecast_demand"].max())

Forecast demand statistics:
count    8.537200e+05
mean     1.796730e-07
std      9.715532e-05
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      6.335764e-02
Name: forecast_demand, dtype: float64

Actual demand statistics:
count    853720.000000
mean          1.386433
std           3.594687
min           0.000000
25%           0.000000
50%           0.000000
75%           1.000000
max         204.000000
Name: actual_demand, dtype: float64

Sample non-zero forecasts:
                item_id store_id       date  actual_demand  forecast_demand
839568  HOUSEHOLD_2_466     TX_1 2016-04-13            0.0         0.000496
839569  HOUSEHOLD_2_466     TX_1 2016-04-14            0.0         0.048986
839570  HOUSEHOLD_2_466     TX_1 2016-04-15            1.0         0.040551
839571  HOUSEHOLD_2_466     TX_1 2016-04-16            0.0         0.063358

Number of non-zero forecasts: 4

Maximum forecast: 0.0633576363325119


In [7]:
# Calculate forecast error statistics for each item-store series

error_stats = (
    forecast_df
    .groupby(["item_id", "store_id"])
    .agg(
        mean_actual_demand=("actual_demand", "mean"),
        std_actual_demand=("actual_demand", "std"),
        mean_forecast_demand=("forecast_demand", "mean"),
        mean_forecast_error=("forecast_error", "mean"),
        std_forecast_error=("forecast_error", "std"),
        mean_absolute_error=("absolute_error", "mean")
    )
    .reset_index()
)

# Series with only one observation can have NaN std.
error_stats["std_actual_demand"] = (
    error_stats["std_actual_demand"].fillna(0)
)

error_stats["std_forecast_error"] = (
    error_stats["std_forecast_error"].fillna(0)
)

print("Error statistics calculated.")

print("\nShape:", error_stats.shape)

print("\nFirst 10 rows:")
print(error_stats.head(10))

print("\nMissing values:")
print(error_stats.isna().sum())

print("\nForecast error standard deviation summary:")
print(error_stats["std_forecast_error"].describe())

Error statistics calculated.

Shape: (30490, 8)

First 10 rows:
       item_id store_id  mean_actual_demand  std_actual_demand  \
0  FOODS_1_001     CA_1            1.178571           1.278123   
1  FOODS_1_001     CA_2            1.107143           2.766724   
2  FOODS_1_001     CA_3            0.857143           2.534419   
3  FOODS_1_001     CA_4            0.321429           0.611832   
4  FOODS_1_001     TX_1            0.035714           0.188982   
5  FOODS_1_001     TX_2            0.392857           0.831744   
6  FOODS_1_001     TX_3            0.500000           0.638285   
7  FOODS_1_001     WI_1            0.607143           1.547741   
8  FOODS_1_001     WI_2            0.392857           0.831744   
9  FOODS_1_001     WI_3            0.357143           0.826160   

   mean_forecast_demand  mean_forecast_error  std_forecast_error  \
0                   0.0             1.178571            1.278123   
1                   0.0             1.107143            2.766724   
2    

In [8]:
# Inventory policy assumptions

LEAD_TIME_DAYS = 7
SERVICE_LEVEL = 0.95
Z_VALUE = 1.645

# Calculate safety stock
error_stats["safety_stock"] = (
    Z_VALUE
    * error_stats["std_forecast_error"]
    * np.sqrt(LEAD_TIME_DAYS)
)

print("Inventory assumptions:")
print("Lead time:", LEAD_TIME_DAYS, "days")
print("Service level:", SERVICE_LEVEL)
print("Z-value:", Z_VALUE)

print("\nSafety stock calculated.")

print("\nSafety stock summary:")
print(error_stats["safety_stock"].describe())

print("\nFirst 10 rows:")
print(
    error_stats[
        [
            "item_id",
            "store_id",
            "std_forecast_error",
            "safety_stock"
        ]
    ].head(10)
)

Inventory assumptions:
Lead time: 7 days
Service level: 0.95
Z-value: 1.645

Safety stock calculated.

Safety stock summary:
count    30490.000000
mean         5.480898
std          6.760612
min          0.000000
25%          2.255288
50%          3.785751
75%          6.307819
max        173.753326
Name: safety_stock, dtype: float64

First 10 rows:
       item_id store_id  std_forecast_error  safety_stock
0  FOODS_1_001     CA_1            1.278123      5.562724
1  FOODS_1_001     CA_2            2.766724     12.041504
2  FOODS_1_001     CA_3            2.534419     11.030453
3  FOODS_1_001     CA_4            0.611832      2.662853
4  FOODS_1_001     TX_1            0.188982      0.822500
5  FOODS_1_001     TX_2            0.831744      3.619969
6  FOODS_1_001     TX_3            0.638285      2.777982
7  FOODS_1_001     WI_1            1.547741      6.736173
8  FOODS_1_001     WI_2            0.831744      3.619969
9  FOODS_1_001     WI_3            0.826160      3.595662


In [9]:
# Calculate 7-day forecasted demand for each item-store series

forecast_df["lead_time_demand"] = (
    forecast_df
    .groupby(["item_id", "store_id"])["forecast_demand"]
    .transform(
        lambda x: x.rolling(
            window=LEAD_TIME_DAYS,
            min_periods=LEAD_TIME_DAYS
        ).sum()
    )
)

print("Lead-time demand calculated.")

print("\nLead-time demand summary:")
print(forecast_df["lead_time_demand"].describe())

print("\nFirst 15 rows:")
print(
    forecast_df[
        [
            "item_id",
            "store_id",
            "date",
            "forecast_demand",
            "lead_time_demand"
        ]
    ].head(15)
)

print("\nMissing lead-time demand values:",
      forecast_df["lead_time_demand"].isna().sum())

Lead-time demand calculated.

Lead-time demand summary:
count    670780.000000
mean          0.000002
std           0.000461
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max           0.153390
Name: lead_time_demand, dtype: float64

First 15 rows:
        item_id store_id       date  forecast_demand  lead_time_demand
0   FOODS_1_001     CA_1 2016-03-28              0.0               NaN
1   FOODS_1_001     CA_1 2016-03-29              0.0               NaN
2   FOODS_1_001     CA_1 2016-03-30              0.0               NaN
3   FOODS_1_001     CA_1 2016-03-31              0.0               NaN
4   FOODS_1_001     CA_1 2016-04-01              0.0               NaN
5   FOODS_1_001     CA_1 2016-04-02              0.0               NaN
6   FOODS_1_001     CA_1 2016-04-03              0.0               0.0
7   FOODS_1_001     CA_1 2016-04-04              0.0               0.0
8   FOODS_1_001     CA_1 2016-04-05              0.0              

##Training-period Demand Statistics

In [11]:
import pyarrow.parquet as pq

DATA_PATH = "/content/drive/MyDrive/features_event_snap.parquet"

TRAIN_END_DATE = pd.Timestamp("2016-03-27")

# Parquet file ko row-group wise process karenge
pf = pq.ParquetFile(DATA_PATH)

print("Number of row groups:", pf.num_row_groups)

# Har item-store series ke liye:
# total demand aur number of training days collect karenge
demand_stats = {}

for rg_idx in range(pf.num_row_groups):

    table = pf.read_row_group(
        rg_idx,
        columns=["item_id", "store_id", "date", "sales"]
    )

    df_rg = table.to_pandas()

    df_rg["date"] = pd.to_datetime(df_rg["date"])

    # Training period only
    df_rg = df_rg[df_rg["date"] <= TRAIN_END_DATE]

    if df_rg.empty:
        del table, df_rg
        continue

    # Aggregate demand within this row group
    grouped = (
        df_rg
        .groupby(["item_id", "store_id"])["sales"]
        .agg(["sum", "count"])
    )

    for (item_id, store_id), row in grouped.iterrows():

        key = (item_id, store_id)

        if key not in demand_stats:
            demand_stats[key] = {
                "total_demand": 0.0,
                "total_days": 0
            }

        demand_stats[key]["total_demand"] += row["sum"]
        demand_stats[key]["total_days"] += row["count"]

    del table, df_rg, grouped

    if (rg_idx + 1) % 10 == 0:
        print(f"Processed row groups: {rg_idx + 1}/{pf.num_row_groups}")

print("\nTraining demand aggregation completed.")
print("Unique item-store series:", len(demand_stats))

Number of row groups: 61
Processed row groups: 10/61
Processed row groups: 20/61
Processed row groups: 30/61
Processed row groups: 40/61
Processed row groups: 50/61
Processed row groups: 60/61

Training demand aggregation completed.
Unique item-store series: 30490


##Average Daily Demand

In [12]:
# Convert aggregated demand statistics into a DataFrame

demand_stats_df = pd.DataFrame([
    {
        "item_id": key[0],
        "store_id": key[1],
        "total_demand": value["total_demand"],
        "total_days": value["total_days"]
    }
    for key, value in demand_stats.items()
])

# Average daily demand
demand_stats_df["average_daily_demand"] = (
    demand_stats_df["total_demand"]
    / demand_stats_df["total_days"]
)

print("Demand statistics DataFrame created.")

print("\nShape:", demand_stats_df.shape)

print("\nFirst 10 rows:")
print(demand_stats_df.head(10))

print("\nMissing values:")
print(demand_stats_df.isna().sum())

print("\nAverage daily demand summary:")
print(demand_stats_df["average_daily_demand"].describe())

print("\nTotal historical days summary:")
print(demand_stats_df["total_days"].describe())

Demand statistics DataFrame created.

Shape: (30490, 5)

First 10 rows:
         item_id store_id  total_demand  total_days  average_daily_demand
0  HOBBIES_1_001     CA_1         573.0        1885              0.303979
1  HOBBIES_1_002     CA_1         491.0        1885              0.260477
2  HOBBIES_1_003     CA_1         272.0        1885              0.144297
3  HOBBIES_1_004     CA_1        3237.0        1885              1.717241
4  HOBBIES_1_005     CA_1        1811.0        1885              0.960743
5  HOBBIES_1_006     CA_1        1625.0        1885              0.862069
6  HOBBIES_1_007     CA_1         413.0        1885              0.219098
7  HOBBIES_1_008     CA_1       13602.0        1885              7.215915
8  HOBBIES_1_009     CA_1        2244.0        1885              1.190451
9  HOBBIES_1_010     CA_1        1357.0        1885              0.719894

Missing values:
item_id                 0
store_id                0
total_demand            0
total_days         

##Reorder Point (ROP)

In [13]:
# Merge historical demand statistics with safety stock

inventory_policy = demand_stats_df.merge(
    error_stats[
        [
            "item_id",
            "store_id",
            "std_forecast_error",
            "safety_stock"
        ]
    ],
    on=["item_id", "store_id"],
    how="left"
)

# Calculate demand expected during lead time
inventory_policy["lead_time_demand"] = (
    inventory_policy["average_daily_demand"]
    * LEAD_TIME_DAYS
)

# Reorder Point
inventory_policy["reorder_point"] = (
    inventory_policy["lead_time_demand"]
    + inventory_policy["safety_stock"]
)

print("Inventory policy calculated.")

print("\nShape:", inventory_policy.shape)

print("\nFirst 10 rows:")
print(
    inventory_policy[
        [
            "item_id",
            "store_id",
            "average_daily_demand",
            "lead_time_demand",
            "safety_stock",
            "reorder_point"
        ]
    ].head(10)
)

print("\nMissing values:")
print(
    inventory_policy[
        [
            "average_daily_demand",
            "lead_time_demand",
            "safety_stock",
            "reorder_point"
        ]
    ].isna().sum()
)

print("\nReorder Point summary:")
print(inventory_policy["reorder_point"].describe())

Inventory policy calculated.

Shape: (30490, 9)

First 10 rows:
         item_id store_id  average_daily_demand  lead_time_demand  \
0  HOBBIES_1_001     CA_1              0.303979          2.127851   
1  HOBBIES_1_002     CA_1              0.260477          1.823342   
2  HOBBIES_1_003     CA_1              0.144297          1.010080   
3  HOBBIES_1_004     CA_1              1.717241         12.020690   
4  HOBBIES_1_005     CA_1              0.960743          6.725199   
5  HOBBIES_1_006     CA_1              0.862069          6.034483   
6  HOBBIES_1_007     CA_1              0.219098          1.533687   
7  HOBBIES_1_008     CA_1              7.215915         50.511406   
8  HOBBIES_1_009     CA_1              1.190451          8.333156   
9  HOBBIES_1_010     CA_1              0.719894          5.039257   

   safety_stock  reorder_point  
0      4.952738       7.080589  
1      1.141447       2.964789  
2      3.003345       4.013425  
3      8.211280      20.231970  
4      5.58

##Scenario Inventory Position

In [14]:
# Create a simple scenario inventory position
# This is a hypothetical inventory scenario because
# actual on-hand inventory is not available in the dataset.

inventory_policy["initial_inventory"] = (
    2 * inventory_policy["reorder_point"]
)

# Initial inventory status
inventory_policy["inventory_status"] = np.where(
    inventory_policy["initial_inventory"]
    <= inventory_policy["reorder_point"],
    "REORDER",
    "SUFFICIENT"
)

print("Scenario inventory status calculated.")

print("\nInventory status counts:")
print(inventory_policy["inventory_status"].value_counts())

print("\nFirst 10 rows:")
print(
    inventory_policy[
        [
            "item_id",
            "store_id",
            "reorder_point",
            "initial_inventory",
            "inventory_status"
        ]
    ].head(10)
)

Scenario inventory status calculated.

Inventory status counts:
inventory_status
SUFFICIENT    30490
Name: count, dtype: int64

First 10 rows:
         item_id store_id  reorder_point  initial_inventory inventory_status
0  HOBBIES_1_001     CA_1       7.080589          14.161179       SUFFICIENT
1  HOBBIES_1_002     CA_1       2.964789           5.929578       SUFFICIENT
2  HOBBIES_1_003     CA_1       4.013425           8.026850       SUFFICIENT
3  HOBBIES_1_004     CA_1      20.231970          40.463939       SUFFICIENT
4  HOBBIES_1_005     CA_1      12.308155          24.616311       SUFFICIENT
5  HOBBIES_1_006     CA_1      10.822597          21.645195       SUFFICIENT
6  HOBBIES_1_007     CA_1       4.001187           8.002373       SUFFICIENT
7  HOBBIES_1_008     CA_1      95.799003         191.598007       SUFFICIENT
8  HOBBIES_1_009     CA_1      16.040768          32.081535       SUFFICIENT
9  HOBBIES_1_010     CA_1       8.795107          17.590214       SUFFICIENT


##Inventory Depletion & Reorder Signal

In [15]:
# Simulate inventory depletion during the validation period
# without replenishment.
#
# This is a scenario analysis because actual inventory
# and replenishment quantities are not available in the dataset.

simulation_df = forecast_df.merge(
    inventory_policy[
        [
            "item_id",
            "store_id",
            "reorder_point",
            "initial_inventory"
        ]
    ],
    on=["item_id", "store_id"],
    how="left"
)

# Cumulative actual demand for each item-store series
simulation_df["cumulative_actual_demand"] = (
    simulation_df
    .groupby(["item_id", "store_id"])["actual_demand"]
    .cumsum()
)

# Inventory remaining after daily demand
simulation_df["inventory_remaining"] = (
    simulation_df["initial_inventory"]
    - simulation_df["cumulative_actual_demand"]
)

# Inventory cannot be negative
simulation_df["inventory_remaining"] = (
    simulation_df["inventory_remaining"].clip(lower=0)
)

# Reorder signal
simulation_df["reorder_signal"] = (
    simulation_df["inventory_remaining"]
    <= simulation_df["reorder_point"]
)

# Inventory status
simulation_df["inventory_status"] = np.where(
    simulation_df["reorder_signal"],
    "REORDER",
    "SUFFICIENT"
)

print("Inventory depletion simulation completed.")

print("\nSimulation shape:", simulation_df.shape)

print("\nInventory status counts:")
print(simulation_df["inventory_status"].value_counts())

print("\nTotal reorder signals:",
      simulation_df["reorder_signal"].sum())

print("\nPercentage of observations requiring reorder:",
      simulation_df["reorder_signal"].mean() * 100)

Inventory depletion simulation completed.

Simulation shape: (853720, 14)

Inventory status counts:
inventory_status
REORDER       439067
SUFFICIENT    414653
Name: count, dtype: int64

Total reorder signals: 439067

Percentage of observations requiring reorder: 51.429859907229535


In [16]:
# Analyze reorder behavior for each item-store series

reorder_analysis = (
    simulation_df
    .groupby(["item_id", "store_id"])
    .agg(
        reorder_point=("reorder_point", "first"),
        initial_inventory=("initial_inventory", "first"),
        minimum_inventory=("inventory_remaining", "min"),
        total_actual_demand=("actual_demand", "sum"),
        reorder_days=("reorder_signal", "sum")
    )
    .reset_index()
)

# First date when inventory reaches reorder point
first_reorder_dates = (
    simulation_df[simulation_df["reorder_signal"]]
    .groupby(["item_id", "store_id"])["date"]
    .min()
    .reset_index()
    .rename(columns={"date": "first_reorder_date"})
)

reorder_analysis = reorder_analysis.merge(
    first_reorder_dates,
    on=["item_id", "store_id"],
    how="left"
)

# Whether the series ever reached reorder point
reorder_analysis["reorder_required"] = (
    reorder_analysis["first_reorder_date"].notna()
)

print("Reorder analysis completed.")

print("\nShape:", reorder_analysis.shape)

print("\nSeries requiring reorder:")
print(
    reorder_analysis["reorder_required"]
    .value_counts()
)

print("\nPercentage of series requiring reorder:",
      reorder_analysis["reorder_required"].mean() * 100)

print("\nFirst 10 rows:")
print(reorder_analysis.head(10))

print("\nReorder days summary:")
print(reorder_analysis["reorder_days"].describe())

Reorder analysis completed.

Shape: (30490, 9)

Series requiring reorder:
reorder_required
True     26931
False     3559
Name: count, dtype: int64

Percentage of series requiring reorder: 88.32732043292883

First 10 rows:
       item_id store_id  reorder_point  initial_inventory  minimum_inventory  \
0  FOODS_1_001     CA_1      11.021610          22.043219           0.000000   
1  FOODS_1_001     CA_2      20.148135          40.296270           9.296270   
2  FOODS_1_001     CA_3      19.489869          38.979738          14.979738   
3  FOODS_1_001     CA_4       5.184339          10.368678           1.368678   
4  FOODS_1_001     TX_1       4.974224           9.948448           8.948448   
5  FOODS_1_001     TX_2       7.708563          15.417126           4.417126   
6  FOODS_1_001     TX_3       5.641111          11.282223           0.000000   
7  FOODS_1_001     WI_1      10.705934          21.411868           4.411868   
8  FOODS_1_001     WI_2       6.828457          13.656913 

In [17]:
# Phase 11 summary

total_series = len(reorder_analysis)

series_requiring_reorder = (
    reorder_analysis["reorder_required"].sum()
)

series_not_requiring_reorder = (
    total_series - series_requiring_reorder
)

reorder_rate = (
    series_requiring_reorder / total_series * 100
)

summary = pd.DataFrame({
    "Metric": [
        "Total item-store series",
        "Lead time",
        "Service level",
        "Average daily demand",
        "Average safety stock",
        "Average reorder point",
        "Series reaching reorder point",
        "Series not reaching reorder point",
        "Reorder-point reach rate"
    ],
    "Value": [
        total_series,
        f"{LEAD_TIME_DAYS} days",
        f"{SERVICE_LEVEL * 100:.0f}%",
        demand_stats_df["average_daily_demand"].mean(),
        inventory_policy["safety_stock"].mean(),
        inventory_policy["reorder_point"].mean(),
        series_requiring_reorder,
        series_not_requiring_reorder,
        f"{reorder_rate:.2f}%"
    ]
})

print("Phase 11 summary:")
print(summary.to_string(index=False))

Phase 11 summary:
                           Metric      Value
          Total item-store series      30490
                        Lead time     7 days
                    Service level        95%
             Average daily demand   1.122458
             Average safety stock   5.480898
            Average reorder point  13.338107
    Series reaching reorder point      26931
Series not reaching reorder point       3559
         Reorder-point reach rate     88.33%


# Phase 11 — Inventory Management

## Objective

This stage converts demand forecasting information into inventory management metrics and replenishment signals.

The analysis calculates:

- Historical average daily demand
- Demand during lead time
- Safety stock
- Reorder Point (ROP)
- Inventory depletion
- Reorder signals

The inventory analysis is performed at the `item_id + store_id` level.

---

## Data Used

The inventory analysis uses:

- Historical sales data from `features_event_snap.parquet`
- Transformer validation forecasts
- Validation-period actual demand
- Forecast error statistics

The historical demand statistics use the training period:

**2011-01-29 to 2016-03-27**

The forecasting validation period is:

**2016-03-28 to 2016-04-24**

There are **30,490 item-store series**.

---

## Inventory Policy Assumptions

Because the dataset does not contain actual inventory levels, supplier lead times, replenishment quantities, or service-level targets, a scenario-based inventory policy is used.

| Parameter | Value |
|---|---:|
| Lead Time | 7 days |
| Service Level | 95% |
| Z-value | 1.645 |

These assumptions are used for demonstration and backtesting purposes and should be replaced with business-specific values in a production system.

---

## Average Daily Demand

For each item-store series:

\[
Average\ Daily\ Demand =
\frac{Total\ Historical\ Demand}{Number\ of\ Training\ Days}
\]

The training period contains 1,885 days for every item-store series.

The average daily demand across all series is:

**1.1225 units/day**

---

## Safety Stock

Safety stock is calculated using forecast error variability:

\[
Safety\ Stock =
Z \times \sigma_{error} \times \sqrt{Lead\ Time}
\]

where:

- \(Z = 1.645\) for a 95% service level
- \(\sigma_{error}\) is the standard deviation of forecast error
- Lead Time = 7 days

Results:

- Average Safety Stock: **5.4809 units**
- Median Safety Stock: **3.7858 units**
- 75th percentile: **6.3078 units**
- Maximum: **173.7533 units**

---

## Reorder Point

The Reorder Point is calculated as:

\[
ROP =
Lead\text{-}Time\ Demand + Safety\ Stock
\]

Lead-time demand is:

\[
Lead\text{-}Time\ Demand =
Average\ Daily\ Demand \times Lead\ Time
\]

Results:

- Average ROP: **13.3381 units**
- Median ROP: **7.0830 units**
- 75th percentile: **13.6865 units**
- Maximum: **1070.1892 units**

---

## Inventory Depletion Scenario

Actual inventory quantities are not available in the dataset.

Therefore, a hypothetical initial inventory level was defined as:

\[
Initial\ Inventory = 2 \times ROP
\]

The inventory was then depleted using realized validation-period demand without adding replenishment orders.

A reorder signal is generated when:

\[
Inventory\ Remaining \leq ROP
\]

This is a scenario analysis rather than an observed inventory policy.

---

## Reorder Analysis Results

Out of 30,490 item-store series:

- **26,931 series (88.33%)** reached their reorder point.
- **3,559 series (11.67%)** did not reach their reorder point.
- Average number of reorder-signal days per series: **14.4 days**
- Median reorder-signal days: **16 days**
- Maximum reorder-signal days: **28 days**

The 88.33% figure represents the percentage of series reaching the reorder point under the hypothetical no-replenishment scenario. It should not be interpreted as the actual percentage of products requiring replenishment in a real business environment.

---

## Key Findings

1. Inventory requirements vary substantially across item-store combinations.
2. Higher historical demand and higher demand uncertainty result in higher reorder points.
3. Safety stock provides an additional buffer against demand uncertainty.
4. The 7-day lead-time assumption converts average daily demand into expected demand during replenishment lead time.
5. The depletion simulation demonstrates how inventory can move toward the reorder point when replenishment is not performed.
6. Actual inventory quantities, supplier lead times, ordering costs, holding costs, and replenishment quantities are required for a production-ready inventory policy.

---

## Limitations

This stage is a scenario-based inventory analysis because the original dataset does not provide:

- Actual on-hand inventory
- Purchase orders
- Supplier lead times
- Replenishment quantities
- Holding costs
- Ordering costs
- Stockout costs
- Service-level targets by product

Therefore, the calculated Safety Stock and ROP should be treated as analytical estimates rather than operational inventory recommendations.

---

## Phase 11 Conclusion

Phase 11 successfully transformed demand and forecast-error information into inventory management metrics.

The final pipeline is:

Historical Demand  
↓  
Average Daily Demand  
↓  
Lead-Time Demand  
↓  
Safety Stock  
↓  
Reorder Point  
↓  
Inventory Depletion  
↓  
Reorder Signal

**Phase 11 — Inventory Management: COMPLETE**